In [7]:
name = "Dipsy Desai"
date = "06/18/2026"

In [8]:
# Setup: shared utilities and tier definitions
import bz2
import urllib.request
import gzip
import io
import json
import zipfile
from contextlib import contextmanager
from pathlib import Path
from IPython.display import Markdown, display


@contextmanager
def open_safe(filename, encoding="utf-8"):
    """Open a file for reading, transparently handling .gz, .bz2, and .zip compression."""
    path = Path(filename)
    suffix = path.suffix.lower()
    if suffix == ".gz":
        with gzip.open(path, "rt", encoding=encoding) as f:
            yield f
    elif suffix == ".bz2":
        with bz2.open(path, "rt", encoding=encoding) as f:
            yield f
    elif suffix == ".zip":
        with zipfile.ZipFile(path) as zf:
            with zf.open(zf.namelist()[0]) as raw:
                yield io.TextIOWrapper(raw, encoding=encoding)
    else:
        with path.open(encoding=encoding) as f:
            yield f


TIERS = [
    ("edge",           1,     1),
    ("transit small",  2,     10),
    ("transit middle", 11,    1000),
    ("transit large",  1001,  10000),
    ("transit huge",   10001, -1),
]

In [9]:
AS_CONE_URL = "http://rook-ceph-rgw-nautiluss3.rook/caida/as-relationships/20260501.ppdc-ases.txt.bz2"
AS_CONE_PATH = Path("data/20260501.ppdc-ases.txt.bz2")

def classify(size: int) -> str:
    for label, lo, hi in TIERS:
        if hi == -1:
            if size >= lo:
                return label
        elif lo <= size <= hi:
            return label
    return "unknown"

print(f"Downloading {AS_CONE_URL} ...")
AS_CONE_PATH.parent.mkdir(parents=True, exist_ok=True)
urllib.request.urlretrieve(AS_CONE_URL, AS_CONE_PATH)

if not AS_CONE_PATH.exists():
    print(f"Unable to save file {AS_CONE_PATH}")
    exit()

# Build a dict mapping each ASN to its customer cone size
asn_to_size = {}

with open_safe(AS_CONE_PATH) as fin:
    for line in fin:
        line = line.strip()

        if not line or line.startswith("#"):
            continue

        fields = line.split()
        asn = fields[0]

        # cone size = number of tokens - 1
        asn_to_size[asn] = len(fields) - 1

print(f"Number of ASNs: {len(asn_to_size)}")

total = len(asn_to_size)
max_size = max(asn_to_size.values()) if asn_to_size else 0

tier_counts = {label: 0 for label, _, _ in TIERS}
for size in asn_to_size.values():
    tier_counts[classify(size)] += 1

header = "| tier | range | number of ASNs | percentage |"
sep = "| ---: | ----- | ---: | ---: |"
table_1_rows = [header, sep]

for label, lo, hi in TIERS:
    if hi == -1:
        range_str = f"{lo}..{max_size}"
    elif lo == hi:
        range_str = str(lo)
    else:
        range_str = f"{lo}..{hi}"

    count = tier_counts[label]
    pct = (count / total * 100) if total else 0

    table_1_rows.append(
        f"| {label} | {range_str} | {count} | {pct:.1f}% |"
    )

table_1 = "\n".join(table_1_rows)

display(Markdown(table_1))
Path("tables").mkdir(exist_ok=True)
Path("tables/asn-cone-tiers.md").write_text(table_1, encoding="utf-8")

Number of ASNs: 79644


| tier | range | number of ASNs | percentage |
| ---: | ----- | ---: | ---: |
| edge | 1 | 67090 | 84.2% |
| transit small | 2..10 | 9977 | 12.5% |
| transit middle | 11..1000 | 2511 | 3.2% |
| transit large | 1001..10000 | 55 | 0.1% |
| transit huge | 10001..56415 | 11 | 0.0% |

279

In [10]:
AS2ORG_URL = "http://rook-ceph-rgw-nautiluss3.rook/caida/as2org/as2org.jsonl"
AS2ORG_PATH = Path("data/as2org.jsonl")

print(f"Downloading {AS2ORG_URL} ...")
AS2ORG_PATH.parent.mkdir(parents=True, exist_ok=True)
urllib.request.urlretrieve(AS2ORG_URL, AS2ORG_PATH)

if not AS2ORG_PATH.exists():
    print(f"Unable to save file {AS2ORG_PATH}")
    exit()

asn_to_country = {}

with open_safe(AS2ORG_PATH) as fin:
    for line in fin:
        line = line.strip()

        if not line:
            continue

        obj = json.loads(line)
        country = obj["country"]

        for asn in obj["members"]:
            asn_to_country[asn] = country

print(f"number of ASNs with countries: {len(asn_to_country)}")

number of ASNs with countries: 120793


In [11]:
display(Markdown(table_1))
Path("tables").mkdir(exist_ok=True)
Path("tables/asn-cone-tiers.md").write_text(table_1, encoding="utf-8")

| tier | range | number of ASNs | percentage |
| ---: | ----- | ---: | ---: |
| edge | 1 | 67090 | 84.2% |
| transit small | 2..10 | 9977 | 12.5% |
| transit middle | 11..1000 | 2511 | 3.2% |
| transit large | 1001..10000 | 55 | 0.1% |
| transit huge | 10001..56415 | 11 | 0.0% |

279

In [26]:
print("""
Parse the AS Customer Cone file and classify each ASN by cone size into tiers (edge, transit small, transit middle, transit large, transit huge). The notebook writes the results to tables/asn-cone-tiers.md.

Question 1: What percentage of ASNs are edge ASes (cone size = 1)? What does this suggest about the structure of the Internet?

Answer 1: 84.2% of ASNs are edge ASes (67,090 out of 79,644). This suggests that the Internet is highly hierarchical, with most networks operating at the edge and serving end users rather than providing transit services. Only a small fraction of ASNs connect and route traffic for other networks.


Question 2: How large is the maximum customer cone? What does this tell you about the most influential ASes on the Internet?

Answer 2: The maximum customer cone size is 56,415. This indicates that a very small number of ASes have enormous reach and influence over Internet connectivity. These ASes sit near the top of the hierarchy and provide connectivity either directly or indirectly to a large portion of the global Internet.


Question 3: How do the proportions of edge, small transit, and large transit ASes compare? What does this distribution reveal about how ASes are organized hierarchically?

Answer 3: Edge ASes account for 84.2% of all ASNs, transit small ASes account for 12.5%, transit middle ASes account for 3.2%, while transit large and transit huge ASes together account for less than 0.1%. This distribution resembles a pyramid: many edge networks exist at the bottom, fewer transit providers exist in the middle, and only a handful of very large providers sit at the top. This demonstrates the hierarchical nature of Internet routing and connectivity.
""")


Parse the AS Customer Cone file and classify each ASN by cone size into tiers (edge, transit small, transit middle, transit large, transit huge). The notebook writes the results to tables/asn-cone-tiers.md.

Question 1: What percentage of ASNs are edge ASes (cone size = 1)? What does this suggest about the structure of the Internet?

Answer 1: 84.2% of ASNs are edge ASes (67,090 out of 79,644). This suggests that the Internet is highly hierarchical, with most networks operating at the edge and serving end users rather than providing transit services. Only a small fraction of ASNs connect and route traffic for other networks.


Question 2: How large is the maximum customer cone? What does this tell you about the most influential ASes on the Internet?

Answer 2: The maximum customer cone size is 56,415. This indicates that a very small number of ASes have enormous reach and influence over Internet connectivity. These ASes sit near the top of the hierarchy and provide connectivity either

In [ ]:
# from collections import defaultdict

# Count ASNs per (tier, country) pair
tier_country_counts = defaultdict(lambda: defaultdict(int))
country_totals = defaultdict(int)

for asn, size in asn_to_size.items():
    tier = classify(size)
    country = asn_to_country.get(asn, "unknown")

    tier_country_counts[tier][country] += 1
    country_totals[country] += 1

# Find top-4 countries by total ASN count
top4 = [c for c, _ in sorted(country_totals.items(),
                             key=lambda x: -x[1])[:4]]

columns = top4 + ["other"]

# Build Markdown table
header = "| tier | " + " | ".join(columns) + " |"
sep = "| --- | " + " | ".join(["---"] * len(columns)) + " |"

table_2_rows = [header, sep]

for label, _, _ in TIERS:
    tier_total = sum(tier_country_counts[label].values())

    row = [label]

    other_count = sum(
        count
        for country, count in tier_country_counts[label].items()
        if country not in top4
    )

    for country in top4:
        n = tier_country_counts[label].get(country, 0)
        pct = (n / tier_total * 100) if tier_total else 0
        row.append(f"{n} ({pct:.1f}%)")

    pct = (other_count / tier_total * 100) if tier_total else 0
    row.append(f"{other_count} ({pct:.1f}%)")

    table_2_rows.append("| " + " | ".join(row) + " |")

table_2 = "\n".join(table_2_rows)

display(Markdown(table_2))

In [25]:
print("""

Join the customer cone tiers with the AS-to-Organization mapping to count tiers per country.

Question 4: Which countries have the most transit huge ASNs? What does this tell you about where Internet infrastructure is concentrated? 

Answer 4: The United States has the most transit huge ASNs, with 7 of the 11 total transit huge ASNs (63.6%). The remaining 4 are spread across other countries. This suggests that major Internet infrastructure and globally influential network providers are concentrated primarily in the United States, reflecting its central role in the development and operation of the global Internet.

Question 5: What proportion of all ASNs fall in the "other" category? What does this suggest about the geographic distribution of the global Internet? 

Answer 5: The "other" category contains more than half of the ASNs in every tier. For example, it contains 56.6% of edge ASes and 55.0% of transit small ASes. This suggests that Internet infrastructure is globally distributed across many countries rather than being concentrated entirely in the four largest countries. While a few countries have large numbers of ASNs, the rest of the world collectively accounts for a significant share of Internet networks.

Question 6: Do the same countries dominate across all AS tiers (edge, transit small, transit huge)? What patterns do you observe across the rows? 

Answer 6: The same countries do not dominate equally across all tiers. The United States consistently has a large share in every tier and strongly dominates the transit huge category. Brazil has a significant presence in the edge and transit small tiers but has no transit huge ASNs. Russia and India also appear in the lower tiers but disappear from the transit huge tier. This pattern suggests that while many countries operate large numbers of local and regional networks, only a few countries host the largest and most globally connected transit providers.

""")



Join the customer cone tiers with the AS-to-Organization mapping to count tiers per country.

Question 4: Which countries have the most transit huge ASNs? What does this tell you about where Internet infrastructure is concentrated? 

Answer 4: The United States has the most transit huge ASNs, with 7 of the 11 total transit huge ASNs (63.6%). The remaining 4 are spread across other countries. This suggests that major Internet infrastructure and globally influential network providers are concentrated primarily in the United States, reflecting its central role in the development and operation of the global Internet.

Question 5: What proportion of all ASNs fall in the "other" category? What does this suggest about the geographic distribution of the global Internet? 

Answer 5: The "other" category contains more than half of the ASNs in every tier. For example, it contains 56.6% of edge ASes and 55.0% of transit small ASes. This suggests that Internet infrastructure is globally distribut